# CAREPILOT — Missing Information Detection Benchmark

*Chapter 2 — État de l'art et benchmarking IA · CAREPILOT operational AI*

## 1. Objective

Detect **missing operational information** in tickets (multilabel) and compute completeness / quality scores before escalation or transfer.

## 2. Problem Type


- **Multilabel classification** (12 canonical missing-field types)
- Outputs: `missing_fields[]`, completeness score, quality level (COMPLETE / ADEQUATE / INCOMPLETE / POOR)

## 3. Dataset Description


- **Source:** `ml_training.historical_tickets` (14,000 tickets)
- **Labels:** 12 missing-information types (e.g. MISSING_LANE_NUMBER, VAGUE_DESCRIPTION)
- **Quality distribution:** COMPLETE 33.5%, partial 59.8%, POOR 6.7%
- **Average completeness:** 77.7%

## 4. Preprocessing

Lowercase normalization, TF-IDF (1–2 grams), handcrafted pattern features (error codes, lane/register, timestamps), multilabel binarization.

## 5. Models / Methods Compared


1. **Rule-Based Detector** — interpretable enterprise baseline
2. **Logistic Regression One-vs-Rest** — lightweight multilabel ML
3. **Random Forest One-vs-Rest** — nonlinear multilabel model


This benchmark compares multiple approaches in order to select the most suitable method for integration into CAREPILOT.

## 6. Evaluation Metrics

Micro/Macro Precision · Micro/Macro Recall · Micro/Macro F1 · Hamming Loss · Exact Match Ratio

## 7. Training or Retrieval Pipeline

Pattern + TF-IDF features → multilabel prediction → completeness score aggregation → quality level assignment.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "ml" / "artifacts").exists():
        ROOT = parent
        break

SHARED = ROOT / "ml" / "notebooks" / "_shared"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from benchmark_notebook_helpers import display_plot, find_project_root, plot_results_table

ROOT = find_project_root()
ARTIFACTS = ROOT / "ml" / "artifacts"
PLOTS = ROOT / "ml" / "plots"
REPORTS = ROOT / "ml" / "reports"
MODELS = ARTIFACTS / "models"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

RUN_BENCHMARK = False  # Set True to re-train / re-run the full benchmark pipeline



In [ ]:
BENCHMARK_SCRIPT = ROOT / "ml/benchmarks/missing_information_detection/benchmark.py"
if RUN_BENCHMARK:
    cmd = [sys.executable, str(BENCHMARK_SCRIPT), "--project-root", str(ROOT), *[]]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipping benchmark run (RUN_BENCHMARK=False). Loading saved artifacts.")
    print("Re-run with RUN_BENCHMARK=True or execute:")
    print(f"  python {BENCHMARK_SCRIPT} --project-root {ROOT}")



## 8. Benchmark Execution — Load Results

In [ ]:
metrics_path = ARTIFACTS / "03_missing_information_benchmark_metrics.json"
        payload = json.loads(metrics_path.read_text(encoding="utf-8"))

rows = payload.get("models", [])
results_df = pd.DataFrame(rows)
if "name" in results_df.columns:
    results_df = results_df.rename(columns={"name": "Method"})
display_cols = [c for c in ["Method", "micro_f1", "macro_f1", "hamming_loss", "exact_match_ratio"] if c in results_df.columns]
if display_cols:
    results_df = results_df[display_cols]

        display(results_df)



## 9. Benchmark Results

Final comparison table (report-ready PNG + DataFrame):

In [ ]:
table_png = PLOTS / "03_missing_information_benchmark_results_table.png"
display_plot(table_png)
display_plot(PLOTS / "03_missing_information_model_comparison.png")
display_plot(PLOTS / "03_missing_information_per_label_f1.png")



## 10. Visualizations

Key plots are loaded from `ml/plots/` (generated by the benchmark pipeline). Re-run with `RUN_BENCHMARK=True` to regenerate.

## 11. Interpretation


**Rule-Based Detector** selected for production: competitive Micro F1 (~0.60), zero training cost, fast inference, full explainability. Macro F1 is lower due to rare labels (MISSING_STORE_IDENTIFIER, MISSING_PAYMENT_REFERENCE) — expected in multilabel enterprise data.

## 12. Selected Model / Method

**Selected for CAREPILOT:**

**Rule-Based Detector** — best operational trade-off (explainability + latency + Micro F1).

## 13. Limitations

- Rare labels drive Macro F1 down
- Ground-truth labels include operator ambiguity
- Rule thresholds require periodic calibration

## 14. Generated Artifacts

- `ml/artifacts/03_missing_information_benchmark_metrics.json`
- `ml/plots/03_missing_information_benchmark_results_table.png`
- `ml/reports/03_missing_information_detection_summary.md`